# 01 · Ingesta y Limpieza de Datos
**Proyecto:** Caso Familia Miranda  
**Autor:** Diego Ballesteros  
**Fecha:** 2026

## Objetivo
Cargar, inspeccionar y limpiar todos los archivos fuente del caso de negocio:
- `presupuesto.xlsx` — hojas Agosto 2023 y Septiembre 2023
- `Gastos_Papa_202308.txt` y `Gastos_Papa_202309.txt` — CSV separado por `;`
- `Gastos_Mama_202308.txt` y `Gastos_Mama_202309.txt` — CSV separado por `;`
- `Gastos_Hijo_202309.xlsx` — solo disponible para septiembre

Al finalizar este notebook se exportan los datos limpios a `data/processed/` listos para ser cargados al modelo relacional en el notebook 02.

## Estructura del notebook
1. Setup e importaciones
2. Carga de archivos fuente
3. Inspección y diagnóstico de calidad
4. Limpieza y estandarización
5. Mapeo de categorías
6. Exportación de datos procesados



## 1. Setup e importaciones

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('..')
RAW_DIR  = BASE_DIR / 'data' / 'raw'
PROC_DIR = BASE_DIR / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

_FILAS_EXCLUIR = {
    'Ingresos', 'Egresos Colombia', 'Varios:', 'TOTAL INGRESOS',
    'TOTAL EGRESOS', 'Ahorro Esperado', 'total ahorros'
}


class DataLoader:
    """Carga los archivos fuente del caso Familia Miranda."""

    def __init__(self, raw_dir: Path):
        self.raw_dir = raw_dir

    def load_gasto_txt(self, filename: str, miembro: str, mes: str) -> pd.DataFrame:
        """Carga un archivo TXT de gastos con separador ';' y comillas simples."""
        df = pd.read_csv(
            self.raw_dir / filename,
            sep=';',
            quotechar="'",
            encoding='utf-8'
        )
        df.columns = df.columns.str.strip().str.replace("'", '', regex=False)
        df['miembro']    = miembro
        df['mes_origen'] = mes
        return df

    def load_gasto_xlsx(self, filename: str, miembro: str, mes: str) -> pd.DataFrame:
        """Carga un archivo Excel de gastos."""
        df = pd.read_excel(self.raw_dir / filename, sheet_name='Sheet1')
        df['miembro']    = miembro
        df['mes_origen'] = mes
        return df

    def load_presupuesto_hoja(self, filename: str, hoja: str, mes: str) -> pd.DataFrame:
        """Carga una hoja del presupuesto y retorna solo los rubros validos."""
        df = pd.read_excel(
            self.raw_dir / filename,
            sheet_name=hoja,
            usecols='A:C',
            header=0,
            engine='openpyxl'
        )
        df.columns = ['rubro', 'valor_planeado', 'valor_real']
        df = df[
            df['rubro'].notna() &
            ~df['rubro'].isin(_FILAS_EXCLUIR) &
            df['valor_planeado'].apply(lambda x: isinstance(x, (int, float)))
        ].copy()
        df['mes'] = mes
        return df[['mes', 'rubro', 'valor_planeado']].reset_index(drop=True)

    def load_all_gastos(self) -> pd.DataFrame:
        """Carga y consolida todos los archivos de gastos de todos los miembros."""
        frames = [
            self.load_gasto_txt('Gastos_Papa_202308.txt', 'papa', '2023-08'),
            self.load_gasto_txt('Gastos_Papa_202309.txt', 'papa', '2023-09'),
            self.load_gasto_txt('Gastos_Mama_202308.txt', 'mama', '2023-08'),
            self.load_gasto_txt('Gastos_Mama_202309.txt', 'mama', '2023-09'),
            self.load_gasto_xlsx('Gastos_Hijo_202309.xlsx', 'hijo', '2023-09'),
        ]
        return pd.concat(frames, ignore_index=True)

    def load_all_presupuesto(self) -> pd.DataFrame:
        """Carga el presupuesto de agosto y septiembre."""
        frames = [
            self.load_presupuesto_hoja('presupuesto.xlsx', 'Agosto 2023',     '2023-08'),
            self.load_presupuesto_hoja('presupuesto.xlsx', 'Septiembre 2023', '2023-09'),
        ]
        return pd.concat(frames, ignore_index=True)



## 2. Carga de archivos fuente

### 2.1 Archivos de gastos CSV (Papá y Mamá)

Los archivos `.txt` usan como separador `;` y cada valor viene envuelto en comillas simples `'`

In [2]:
loader = DataLoader(RAW_DIR)

df_gastos_raw = loader.load_all_gastos()
df_ppto_raw   = loader.load_all_presupuesto()

print(f'Gastos cargados     : {len(df_gastos_raw)} registros')
print(f'Presupuesto cargado : {len(df_ppto_raw)} registros')
print()
print('Registros por miembro y mes:')
print(df_gastos_raw.groupby(['miembro', 'mes_origen']).size().to_string())
print()
df_gastos_raw.head(3)


Gastos cargados     : 372 registros
Presupuesto cargado : 74 registros

Registros por miembro y mes:
miembro  mes_origen
hijo     2023-09        56
mama     2023-08        27
         2023-09        46
papa     2023-08       117
         2023-09       126



,fecha,flujo casa mes,valor,Forma de Pago,idCategoria,miembro,mes_origen
0,2023-08-01,Contrato papa,10574000,Debito,Contrato papa,papa,2023-08
1,2023-08-31,Pago gas,8736,Debito,PAGO GAS,papa,2023-08
2,2023-08-31,Pago Agua,42630,Debito,PAGO AGUA,papa,2023-08


---
## 3. Inspección y diagnóstico de calidad

In [3]:
class DataValidator:
    """Ejecuta diagnosticos de calidad sobre el DataFrame de gastos."""

    def __init__(self, df: pd.DataFrame):
        self.df = df

    def summary(self) -> None:
        """Imprime resumen general: total de registros, columnas, nulos y tipos."""
        print('DIAGNOSTICO DE CALIDAD — GASTOS CONSOLIDADOS')
        print('=' * 55)
        print(f'Total registros : {len(self.df)}')
        print(f'Columnas        : {list(self.df.columns)}')
        print()
        print('Valores nulos por columna:')
        print(self.df.isnull().sum().to_string())
        print()
        print('Tipos de datos:')
        print(self.df.dtypes.to_string())

    def check_non_numeric(self) -> pd.DataFrame:
        """Retorna filas con valores no numericos en la columna valor."""
        return self.df[
            pd.to_numeric(self.df['valor'], errors='coerce').isna() &
            self.df['valor'].notna()
        ]

    def check_duplicate_headers(self) -> pd.DataFrame:
        """Retorna filas que parecen encabezados repetidos del CSV."""
        return self.df[
            self.df['fecha'].astype(str).str.contains(
                'fecha|idCategoria', case=False, na=False
            )
        ]

    def distribution(self) -> None:
        """Muestra la distribucion de registros por miembro y mes."""
        print('Distribucion por miembro y mes:')
        print(self.df.groupby(['miembro', 'mes_origen']).size().to_string())


In [4]:
validator = DataValidator(df_gastos_raw)
validator.summary()

print()
invalidos = validator.check_non_numeric()
if len(invalidos) > 0:
    print('VALORES NO NUMERICOS DETECTADOS:')
    print(invalidos[['fecha', 'flujo casa mes', 'valor', 'miembro', 'mes_origen']].to_string())
else:
    print('No se encontraron valores no numericos.')

print()
encabezados = validator.check_duplicate_headers()
print(f'Filas de encabezado repetidas: {len(encabezados)}')

print()
validator.distribution()


DIAGNOSTICO DE CALIDAD — GASTOS CONSOLIDADOS
Total registros : 372
Columnas        : ['fecha', 'flujo casa mes', 'valor', 'Forma de Pago', 'idCategoria', 'miembro', 'mes_origen']

Valores nulos por columna:
fecha             0
flujo casa mes    0
valor             0
Forma de Pago     0
idCategoria       0
miembro           0
mes_origen        0

Tipos de datos:
fecha             object
flujo casa mes       str
valor             object
Forma de Pago        str
idCategoria          str
miembro              str
mes_origen           str

VALORES NO NUMERICOS DETECTADOS:
         fecha  flujo casa mes       valor miembro mes_origen
58  2023-08-12  Mercado Origen  8MIL PESOS    papa    2023-08

Filas de encabezado repetidas: 0

Distribucion por miembro y mes:
miembro  mes_origen
hijo     2023-09        56
mama     2023-08        27
         2023-09        46
papa     2023-08       117
         2023-09       126


## 4. Limpieza y estandarización

Con base en el diagnóstico, se aplican las siguientes correcciones:

| # | Problema | Acción |
|---|----------|--------|
| 1 | Valor `'8MIL PESOS'` no numérico | Imputar `8000` y registrar en log |
| 2 | Filas de encabezado duplicadas en CSV | Eliminar |
| 3 | Columna `fecha` como string | Convertir a `datetime` |
| 4 | Espacios y mayúsculas inconsistentes en categorías | Normalizar |
| 5 | Filas `idCategoria == 'efectivo'` (error de captura) | Eliminar |

In [5]:
class DataCleaner:
    """Limpia y estandariza el DataFrame de gastos."""

    def __init__(self):
        self.log = []

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Aplica todas las transformaciones de limpieza en secuencia."""
        df = df.copy()
        df = self._remove_header_rows(df)
        df = self._remove_efectivo_errors(df)
        df = self._fix_8mil(df)
        df = self._convert_types(df)
        df = self._strip_strings(df)
        df = self._rename_columns(df)
        df = self._remove_duplicates_between_members(df)
        return df

    def _remove_duplicates_between_members(self, df: pd.DataFrame) -> pd.DataFrame:
        """Elimina gastos duplicados entre miembros de la familia.

        Regla: Un gasto con misma fecha, descripcion y valor solo puede
        pertenecer a UN miembro. Se mantiene el primer registro encontrado.
        """
        antes = len(df)
        # Identificar duplicados por fecha, descripcion y valor
        duplicados = df.duplicated(subset=['fecha', 'descripcion', 'valor'], keep='first')
        df = df[~duplicados].copy()
        despues = len(df)
        eliminados = antes - despues
        self.log.append(f'[OK] Duplicados entre miembros eliminados: {eliminados}')
        return df

    def separar_contratos(self, df: pd.DataFrame) -> tuple:
        """Separa contratos (ingresos) del resto de gastos.
        
        Retorna:
            tuple: (df_gastos_sin_contratos, df_contratos)
        """
        # Identificar contratos por id_categoria que empieza con 'Contrato' o 'contrato'
        mask_contratos = df['id_categoria'].str.lower().str.startswith('contrato')
        df_contratos = df[mask_contratos].copy()
        df_gastos = df[~mask_contratos].copy()
        self.log.append(f'[OK] Contratos separados como ingresos: {len(df_contratos)} registro(s)')
        return df_gastos, df_contratos

    def apply_mapeo(self, df: pd.DataFrame, mapeo: dict) -> pd.DataFrame:
        """Agrega rubro_presupuesto usando la tabla de mapeo de categorias."""
        df = df.copy()
        df['rubro_presupuesto'] = df['id_categoria'].map(mapeo)
        return df

    def print_log(self) -> None:
        print('RESUMEN DE LIMPIEZA')
        print('-' * 50)
        for entry in self.log:
            print(f'  {entry}')

    def _remove_header_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        mask = df['fecha'].astype(str).str.contains('fecha|idCategoria', case=False, na=False)
        self.log.append(f'[OK] Filas de encabezado eliminadas: {mask.sum()}')
        return df[~mask].copy()

    def _remove_efectivo_errors(self, df: pd.DataFrame) -> pd.DataFrame:
        mask = df['idCategoria'].astype(str).str.strip().str.lower() == 'efectivo'
        self.log.append(f'[OK] Filas con idCategoria="efectivo" eliminadas: {mask.sum()}')
        return df[~mask].copy()

    def _fix_8mil(self, df: pd.DataFrame) -> pd.DataFrame:
        mask = df['valor'].astype(str).str.upper().str.contains('8MIL', na=False)
        df.loc[mask, 'valor'] = 8000
        self.log.append(f'[IMPUTADO] "8MIL PESOS" -> 8000: {mask.sum()} registro(s) (Papa, Ago 2023)')
        return df

    def _convert_types(self, df: pd.DataFrame) -> pd.DataFrame:
        df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
        df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
        return df

    def _strip_strings(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in ['flujo casa mes', 'Forma de Pago', 'idCategoria']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip()
        return df

    def _rename_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.rename(columns={
            'flujo casa mes' : 'descripcion',
            'Forma de Pago'  : 'forma_pago',
            'idCategoria'    : 'id_categoria',
        })


cleaner   = DataCleaner()
df_gastos_all = cleaner.clean(df_gastos_raw)

# Separar contratos (ingresos) de gastos
df_gastos, df_contratos = cleaner.separar_contratos(df_gastos_all)

cleaner.print_log()
print()
print(f'Registros antes  : {len(df_gastos_raw)}')
print(f'Registros gastos : {len(df_gastos)}')
print(f'Registros contratos (ingresos): {len(df_contratos)}')
print()
print(df_gastos.dtypes.to_string())

RESUMEN DE LIMPIEZA
--------------------------------------------------
  [OK] Filas de encabezado eliminadas: 0
  [OK] Filas con idCategoria="efectivo" eliminadas: 0
  [IMPUTADO] "8MIL PESOS" -> 8000: 1 registro(s) (Papa, Ago 2023)
  [OK] Duplicados entre miembros eliminados: 113
  [OK] Contratos separados como ingresos: 5 registro(s)

Registros antes  : 372
Registros gastos : 254
Registros contratos (ingresos): 5

fecha           datetime64[us]
descripcion                str
valor                  float64
forma_pago                 str
id_categoria               str
miembro                    str
mes_origen                 str


---
## 5. Mapeo de categorías

Las categorías de los archivos de gastos **no coinciden exactamente** con los rubros del presupuesto. Se construye una tabla de equivalencias para poder cruzar ambas fuentes.

Categorías en gastos que **no tienen rubro en el presupuesto** son un hallazgo analítico valioso — responden la pregunta: *¿qué gastos no están en rubros presupuestados?*

In [6]:
# Mapeo normalizado - usa nombres consistentes de rubros
# Se normalizan las inconsistencias del presupuesto original:
# - CUOTA CASA y CUOTA  CASA (doble espacio) -> CUOTA CASA
# - OFRENDAS FAMILIA y AYUDAS A FAMILIARES -> AYUDAS A FAMILIARES
# - ENTRETENIMIENTO/VIAJE y ENTRENAMIENTO/VIAJE -> ENTRETENIMIENTO/VIAJE

MAPEO_CATEGORIAS = {
    # Vivienda y servicios
    'CUOTA CASA'                      : 'CUOTA CASA',  # Normalizado (era CUOTA  CASA con doble espacio)
    'PAGO ADMINISTACION CASA'         : 'PAGO ADMINISTACION CASA',
    'PAGO LUZ'                        : 'PAGO LUZ',
    'PAGO AGUA'                       : 'PAGO AGUA',
    'PAGO GAS'                        : 'PAGO GAS',
    'PAGO CLARO MOVIL'                : 'PAGO CLARO MOVIL',
    'PAGO INTERNET'                   : 'PAGO INTERNET',
    'COSAS DE CASA'                   : 'COSAS DE CASA',
    'JARDIN'                          : 'JARDIN',

    # Salud
    'PAGO SALUD Y PENSIONES'          : 'PAGO SALUD Y PENSIONES',
    'PAGO MEDICINA PREPAGADA'         : 'PAGO MEDICINA PREPAGADA',
    'MEDICINA - MEDICO'               : 'CITAS MEDICAS',

    # Alimentacion
    'MERCADO'                         : 'MERCADO',
    'COMIDAS AFUERA'                  : 'COMIDAS AFUERA',
    'SANCOCHO'                        : 'COMIDAS AFUERA',

    # Transporte
    'TRANSPORTE CARRO / MOTO'         : 'TRANSPORTE CARRO/MOTO/GASOLINA',
    'TRANSPORTE TAXI'                 : 'TRANSPORTE TAXI',
    'VIAJES'                          : 'VIAJES',

    # Finanzas
    'PAGO TARJETA NUBE'               : 'PAGO TARJETA NUBE',
    'CUOTAS DE MANEJO'                : None,   # gasto no presupuestado
    'PRESTAMO'                        : None,   # gasto no presupuestado
    'RETIRO CAJERO'                   : None,   # gasto no presupuestado
    'inversiones'                     : None,   # gasto no presupuestado

    # Familia y religion
    'OFRENDAS FAMILIA'                : 'AYUDAS A FAMILIARES',  # Normalizado

    # Ocio y deporte
    'DEPORTE ENTRENO + INSCRIPCIONES' : 'DEPORTE ENTRENO + INSCRIPCIONES',
    'ENTRETENIMIENTO/VIAJE'           : 'ENTRETENIMIENTO/VIAJE',  # Normalizado

    # Otros
    'DIEZMO'                          : 'DIEZMO',
    'PELUQUERIA'                      : 'PELUQUERIA',
    'ROPA'                            : 'ROPA',
    'REGALOS'                         : 'REGALOS',
    'LIBROS'                          : 'LIBROS',
    'OTROS'                           : 'OTROS',
}

# Nota: Los contratos (Contrato papa, Contrato mama, contrato hijo) se manejan
# como ingresos separados, no como gastos. Por eso NO están en este mapeo.

df_gastos = cleaner.apply_mapeo(df_gastos, MAPEO_CATEGORIAS)

sin_mapeo_inesperado = df_gastos[
    df_gastos['rubro_presupuesto'].isna() &
    ~df_gastos['id_categoria'].isin(['CUOTAS DE MANEJO', 'PRESTAMO', 'RETIRO CAJERO', 'inversiones'])
]

if len(sin_mapeo_inesperado) > 0:
    print('Categorias sin mapeo definido:')
    print(sin_mapeo_inesperado['id_categoria'].value_counts().to_string())
else:
    print('Todas las categorias tienen mapeo definido.')

print()
print('Gastos NO presupuestados (sin rubro):')
print(
    df_gastos[df_gastos['rubro_presupuesto'].isna()]
    .groupby('id_categoria')['valor']
    .agg(['count', 'sum'])
    .rename(columns={'count': '# registros', 'sum': 'total_gasto'})
    .sort_values('total_gasto', ascending=False)
    .to_string()
)

Todas las categorias tienen mapeo definido.

Gastos NO presupuestados (sin rubro):
                  # registros   total_gasto
id_categoria                               
inversiones                 2 39,157,432.00
PRESTAMO                    2  4,000,000.00
RETIRO CAJERO               4  2,000,000.00
CUOTAS DE MANEJO            4    147,449.54


---
## 6. Exportación de datos procesados

Se exportan tres archivos limpios a `data/processed/` que serán la entrada del notebook 02 (modelo relacional).

In [7]:
class DataExporter:
    """Exporta los datos procesados a data/processed/."""

    def __init__(self, proc_dir: Path):
        self.proc_dir = proc_dir

    def export_gastos(self, df: pd.DataFrame) -> None:
        cols = ['fecha', 'descripcion', 'valor', 'forma_pago',
                'id_categoria', 'rubro_presupuesto', 'miembro', 'mes_origen']
        df_out = df[cols].copy()
        # Convertir valor a entero para evitar el .0
        df_out['valor'] = df_out['valor'].astype(int)
        # Convertir fecha a string sin hora
        df_out['fecha'] = pd.to_datetime(df_out['fecha']).dt.strftime('%Y-%m-%d')
        df_out.to_csv(self.proc_dir / 'gastos_clean.csv', index=False)
        print(f'gastos_clean.csv      -> {len(df)} registros')

    def export_ingresos(self, df: pd.DataFrame) -> None:
        """Exporta los ingresos (contratos) separados."""
        cols = ['fecha', 'descripcion', 'valor', 'forma_pago', 'miembro', 'mes_origen']
        df_out = df[cols].copy()
        df_out['valor'] = df_out['valor'].astype(int)
        df_out['fecha'] = pd.to_datetime(df_out['fecha']).dt.strftime('%Y-%m-%d')
        # Agregar columna tipo_ingreso para identificar que son contratos
        df_out['tipo_ingreso'] = 'contrato'
        df_out.to_csv(self.proc_dir / 'ingresos_clean.csv', index=False)
        print(f'ingresos_clean.csv    -> {len(df)} registros')

    def export_presupuesto(self, df: pd.DataFrame) -> None:
        df_out = df.copy()
        # Normalizar nombres de rubros inconsistentes entre meses
        df_out['rubro'] = df_out['rubro'].str.strip()
        # Corregir dobles espacios y nombres inconsistentes
        df_out['rubro'] = df_out['rubro'].replace({
            'CUOTA  CASA': 'CUOTA CASA',
            'CUOTA CASA': 'CUOTA CASA',
            'OFRENDAS FAMILIA': 'AYUDAS A FAMILIARES',
            'ENTRENAMIENTO/VIAJE': 'ENTRETENIMIENTO/VIAJE',
        })
        # Convertir valor a entero
        df_out['valor_planeado'] = df_out['valor_planeado'].astype(int)
        df_out.to_csv(self.proc_dir / 'presupuesto_clean.csv', index=False)
        print(f'presupuesto_clean.csv -> {len(df_out)} registros')

    def export_mapeo(self, mapeo: dict) -> None:
        df_mapeo = pd.DataFrame(
            list(mapeo.items()),
            columns=['categoria_gasto', 'rubro_presupuesto']
        )
        df_mapeo.to_csv(self.proc_dir / 'mapeo_categorias.csv', index=False)
        print(f'mapeo_categorias.csv  -> {len(df_mapeo)} reglas')


exporter = DataExporter(PROC_DIR)
exporter.export_gastos(df_gastos)
exporter.export_ingresos(df_contratos)
exporter.export_presupuesto(df_ppto_raw)
exporter.export_mapeo(MAPEO_CATEGORIAS)

gastos_clean.csv      -> 254 registros
ingresos_clean.csv    -> 5 registros
presupuesto_clean.csv -> 74 registros
mapeo_categorias.csv  -> 32 reglas


In [8]:
print('VERIFICACION DE ARCHIVOS EXPORTADOS')
print('-' * 50)
for f in sorted(PROC_DIR.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f'  {f.name:<40} {size_kb:>6.1f} KB')

print()
print('=' * 60)
print('RESUMEN DE DATOS CORREGIDOS')
print('=' * 60)
print()
print('INGRESOS (Contratos separados):')
print(
    df_contratos.groupby(['miembro', 'mes_origen'])['valor']
    .agg(['count', 'sum'])
    .rename(columns={'count': 'registros', 'sum': 'total'})
    .to_string()
)
print()
print('GASTOS POR MIEMBRO (sin duplicados ni contratos):')
print(
    df_gastos.groupby(['miembro', 'mes_origen'])['valor']
    .agg(['count', 'sum'])
    .rename(columns={'count': 'registros', 'sum': 'total'})
    .to_string()
)
print()
print('GASTOS POR MES (totales corregidos):')
totales = df_gastos.groupby('mes_origen')['valor'].sum()
for mes, total in totales.items():
    print(f'  {mes}: ${total:,.0f} COP')
print()
print('INGRESOS POR MES:')
ingresos = df_contratos.groupby('mes_origen')['valor'].sum()
for mes, total in ingresos.items():
    print(f'  {mes}: ${total:,.0f} COP')

VERIFICACION DE ARCHIVOS EXPORTADOS
--------------------------------------------------
  comparativo_meses.png                      66.7 KB
  deportes_familia.png                       55.9 KB
  ejecucion_presupuestal.png                107.0 KB
  flujo_caja.png                             40.1 KB
  gastos_clean.csv                           20.2 KB
  ingresos_clean.csv                          0.4 KB
  mapeo_categorias.csv                        0.9 KB
  medios_pago.png                            54.9 KB
  presupuesto_clean.csv                       2.1 KB
  top3_sobreejercicio.png                    35.9 KB
  wordcloud_familia.png                     657.0 KB

RESUMEN DE DATOS CORREGIDOS

INGRESOS (Contratos separados):
                    registros         total
miembro mes_origen                         
hijo    2023-09             1  2,500,000.00
mama    2023-08             1  8,500,000.00
        2023-09             1  8,500,000.00
papa    2023-08             1 10,574,000.00
    